## Inicialización de Variables
Esta celda carga las variables necesarias para que el notebook se ejecute correctamente de forma independiente.

In [8]:
from pathlib import Path
import sys

RAIZ_PROYECTO = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from src.corpus_inmuebles import preparar_corpus_para_modelado
from src.infraestructura_cpu import configurar_torch_cpu, relevar_hardware, sugerir_tamanio_muestra
from src.property_text_pipeline import (
    COLUMNA_TEXTO_LIMPIO_TRANSFORMER,
    COLUMNA_TEXTO_LIMPIO_TRANSFORMER_CENSURADO,
)
from src.transformer_cpu import (
    cargar_modelo_transformer_para_clasificacion,
    cargar_tokenizador_transformer,
    construir_mapeo_etiquetas,
    crear_dataloaders_transformer,
    predecir_con_transformer,
)

RUTA_DATOS = RAIZ_PROYECTO / 'data' / 'entrenamiento.csv'
resumen = relevar_hardware()
configurar_torch_cpu()
tamanio_muestra = sugerir_tamanio_muestra(resumen.memoria_disponible_gb)

df_muestra, df_entrenamiento, df_prueba = preparar_corpus_para_modelado(
    ruta_datos=RUTA_DATOS,
    tamanio_muestra=tamanio_muestra,
    tamanio_test=0.2,
    semilla=42,
)

etiqueta_a_id, id_a_etiqueta = construir_mapeo_etiquetas(df_muestra['property_type'])

def _asegurar_predicciones_transformer(nombre_predicciones, nombre_modelo_ram, variante_artifacto, columna_texto):
    if nombre_predicciones in globals():
        return

    try:
        tokenizador_local = cargar_tokenizador_transformer(variante_artifacto=variante_artifacto)
        modelo_local = cargar_modelo_transformer_para_clasificacion(
            cantidad_etiquetas=len(etiqueta_a_id),
            etiqueta_a_id=etiqueta_a_id,
            id_a_etiqueta=id_a_etiqueta,
            variante_artifacto=variante_artifacto,
        )
    except FileNotFoundError as error:
        raise RuntimeError(
            f"No se pudieron reconstruir {nombre_predicciones} desde artifacts/{variante_artifacto}. "
            "Ejecuta 02_fase_4_transformer_cpu.ipynb para generar los snapshots locales."
        ) from error
    _, dataloader_prueba = crear_dataloaders_transformer(
        df_entrenamiento,
        df_prueba,
        tokenizador_local,
        etiqueta_a_id=etiqueta_a_id,
        columna_texto=columna_texto,
        batch_size=4,
        longitud_maxima=128,
    )
    pred_ids = predecir_con_transformer(modelo_local, dataloader_prueba)
    globals()[nombre_predicciones] = [id_a_etiqueta[indice] for indice in pred_ids]
    globals()[nombre_modelo_ram] = modelo_local
    if nombre_modelo_ram == "modelo_censurado":
        globals()["tokenizador"] = tokenizador_local

modelo_normal = globals().get("modelo_normal", globals().get("modelo"))

_asegurar_predicciones_transformer(
    "predicciones",
    "modelo_normal",
    "distilbert_normal",
    COLUMNA_TEXTO_LIMPIO_TRANSFORMER,
)
_asegurar_predicciones_transformer(
    "predicciones_censurado",
    "modelo_censurado",
    "distilbert_censurado_1ep",
    COLUMNA_TEXTO_LIMPIO_TRANSFORMER_CENSURADO,
)
_asegurar_predicciones_transformer(
    "predicciones_censurado_2ep",
    "modelo_censurado_2ep",
    "distilbert_censurado_2ep",
    COLUMNA_TEXTO_LIMPIO_TRANSFORMER_CENSURADO,
)

if "tokenizador" not in globals() and "modelo_censurado" in globals():
    try:
        tokenizador = cargar_tokenizador_transformer(variante_artifacto="distilbert_censurado_1ep")
    except FileNotFoundError:
        pass


In [9]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def metricas_globales(y_true, y_pred, modelo, escenario):
    _ = modelo, escenario
    if y_true is None or y_pred is None:
        return {"accuracy": "N/A", "f1_macro": "N/A"}
    return {
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "f1_macro": round(float(precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)[2]), 4),
    }

def _entrenar_modelos_clasicos_si_falta():
    if "df_prueba" not in globals() or "df_entrenamiento" not in globals(): return
    from src.property_text_pipeline import COLUMNA_OBJETIVO, COLUMNA_TEXTO_LIMPIO, COLUMNA_TEXTO_LIMPIO_CENSURADO, entrenar_modelo_base_svm, entrenar_modelo_bayes, entrenar_modelo_logistica
    if COLUMNA_OBJETIVO not in df_entrenamiento.columns or COLUMNA_OBJETIVO not in df_prueba.columns: return

    etiquetas_entrenamiento = df_entrenamiento[COLUMNA_OBJETIVO]

    def _entrenar_y_guardar(nombre_variable, funcion_entrenamiento, columna_texto):
        if globals().get(nombre_variable, None) is not None: return
        if columna_texto not in df_entrenamiento.columns or columna_texto not in df_prueba.columns: return
        modelo = funcion_entrenamiento(df_entrenamiento, etiquetas_entrenamiento, columna_texto=columna_texto)
        globals()[nombre_variable] = modelo.predict(df_prueba[columna_texto])

    _entrenar_y_guardar("predicciones_bayes", entrenar_modelo_bayes, COLUMNA_TEXTO_LIMPIO)
    _entrenar_y_guardar("predicciones_svm", entrenar_modelo_base_svm, COLUMNA_TEXTO_LIMPIO)
    _entrenar_y_guardar("predicciones_logistica", entrenar_modelo_logistica, COLUMNA_TEXTO_LIMPIO)
    _entrenar_y_guardar("predicciones_bayes_censurado", entrenar_modelo_bayes, COLUMNA_TEXTO_LIMPIO_CENSURADO)
    _entrenar_y_guardar("predicciones_svm_censurado", entrenar_modelo_base_svm, COLUMNA_TEXTO_LIMPIO_CENSURADO)
    _entrenar_y_guardar("predicciones_logistica_censurado", entrenar_modelo_logistica, COLUMNA_TEXTO_LIMPIO_CENSURADO)

def buscar_prediccion(nombre):
    return globals().get(nombre, None)

variables_transformer_requeridas = ["predicciones", "predicciones_censurado", "predicciones_censurado_2ep"]
faltantes_transformer = [nombre for nombre in variables_transformer_requeridas if nombre not in globals()]
if faltantes_transformer:
    raise RuntimeError(
        "Faltan predicciones del transformer en la sesion actual: "
        + ", ".join(faltantes_transformer)
        + ". Ejecuta la celda de preparacion de este notebook o vuelve a generar los artefactos con 02_fase_4_transformer_cpu.ipynb."
    )

y_real = df_prueba["property_type"] if "df_prueba" in globals() else None
if y_real is not None:
    _entrenar_modelos_clasicos_si_falta()

modelos = [
    ("Bayes", "Base", "predicciones_bayes"),
    ("SVM", "Base", "predicciones_svm"),
    ("Reg Log", "Base", "predicciones_logistica"),
    ("DistilBERT 1 epoch", "Base", "predicciones"),
    ("Bayes", "Censurado", "predicciones_bayes_censurado"),
    ("SVM", "Censurado", "predicciones_svm_censurado"),
    ("Reg Log", "Censurado", "predicciones_logistica_censurado"),
    ("DistilBERT 1 epoch", "Censurado", "predicciones_censurado"),
    ("DistilBERT 2 epoch", "Censurado", "predicciones_censurado_2ep"),
]

tabla = []
for modelo, escenario, variable in modelos:
    y_pred = buscar_prediccion(variable)
    tabla.append({
        "Modelo": modelo,
        "Escenario": escenario,
        **metricas_globales(y_real, y_pred, modelo, escenario),
    })

_df_tabla1 = pd.DataFrame(tabla)[["Modelo", "Escenario", "accuracy", "f1_macro"]]
df_tabla1 = _df_tabla1.rename(columns={"accuracy": "Accuracy global", "f1_macro": "F1 macro"})
print("Tabla 1: Resumen global de métricas")
display(df_tabla1)
if y_real is None:
    print("⚠️ df_prueba no está definido en el espacio de ejecución. Ejecuta previamente las celdas de carga y evaluación.")


Tabla 1: Resumen global de métricas


,Modelo,Escenario,Accuracy global,F1 macro
0,Bayes,Base,0.7850,0.6799
1,SVM,Base,0.9040,0.8129
2,Reg Log,Base,0.9085,0.8161
3,DistilBERT 1 epoch,Base,0.9035,0.8292
4,Bayes,Censurado,0.7555,0.6538
5,SVM,Censurado,0.8355,0.7262
6,Reg Log,Censurado,0.8515,0.7483
7,DistilBERT 1 epoch,Censurado,0.8180,0.7038
8,DistilBERT 2 epoch,Censurado,0.8165,0.7075


In [10]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

def f1_por_clase(y_true, y_pred, etiquetas, modelo, escenario):
    _ = modelo, escenario
    if y_true is None or y_pred is None:
        return {f"F1 {etiq}": "N/A" for etiq in etiquetas}

    _, _, f1_scores, _ = precision_recall_fscore_support(y_true, y_pred, labels=etiquetas, average=None, zero_division=0)
    return {f"F1 {etiq}": round(float(v), 4) for etiq, v in zip(etiquetas, f1_scores)}

def _entrenar_modelos_clasicos_si_falta():
    if "df_prueba" not in globals() or "df_entrenamiento" not in globals(): return
    from src.property_text_pipeline import COLUMNA_OBJETIVO, COLUMNA_TEXTO_LIMPIO, COLUMNA_TEXTO_LIMPIO_CENSURADO, entrenar_modelo_base_svm, entrenar_modelo_bayes, entrenar_modelo_logistica
    if COLUMNA_OBJETIVO not in df_entrenamiento.columns or COLUMNA_OBJETIVO not in df_prueba.columns: return

    etiquetas_entrenamiento = df_entrenamiento[COLUMNA_OBJETIVO]

    def _entrenar_y_guardar(nombre_variable, funcion_entrenamiento, columna_texto):
        if globals().get(nombre_variable, None) is not None: return
        if columna_texto not in df_entrenamiento.columns or columna_texto not in df_prueba.columns: return
        modelo = funcion_entrenamiento(df_entrenamiento, etiquetas_entrenamiento, columna_texto=columna_texto)
        globals()[nombre_variable] = modelo.predict(df_prueba[columna_texto])

    _entrenar_y_guardar("predicciones_bayes", entrenar_modelo_bayes, COLUMNA_TEXTO_LIMPIO)
    _entrenar_y_guardar("predicciones_svm", entrenar_modelo_base_svm, COLUMNA_TEXTO_LIMPIO)
    _entrenar_y_guardar("predicciones_logistica", entrenar_modelo_logistica, COLUMNA_TEXTO_LIMPIO)
    _entrenar_y_guardar("predicciones_bayes_censurado", entrenar_modelo_bayes, COLUMNA_TEXTO_LIMPIO_CENSURADO)
    _entrenar_y_guardar("predicciones_svm_censurado", entrenar_modelo_base_svm, COLUMNA_TEXTO_LIMPIO_CENSURADO)
    _entrenar_y_guardar("predicciones_logistica_censurado", entrenar_modelo_logistica, COLUMNA_TEXTO_LIMPIO_CENSURADO)

variables_transformer_requeridas = ["predicciones", "predicciones_censurado", "predicciones_censurado_2ep"]
faltantes_transformer = [nombre for nombre in variables_transformer_requeridas if nombre not in globals()]
if faltantes_transformer:
    raise RuntimeError(
        "Faltan predicciones del transformer en la sesion actual: "
        + ", ".join(faltantes_transformer)
        + ". Ejecuta la celda de preparacion de este notebook o vuelve a generar los artefactos con 02_fase_4_transformer_cpu.ipynb."
    )

etiquetas_clase = ["Casa", "Departamento", "PH"]
y_real = df_prueba["property_type"] if "df_prueba" in globals() else None
if y_real is not None:
    _entrenar_modelos_clasicos_si_falta()

modelos_a_evaluar = [
    ("Bayes", "Base", "predicciones_bayes"),
    ("SVM", "Base", "predicciones_svm"),
    ("Reg Log", "Base", "predicciones_logistica"),
    ("DistilBERT 1 epoch", "Base", "predicciones"),
    ("Bayes", "Censurado", "predicciones_bayes_censurado"),
    ("SVM", "Censurado", "predicciones_svm_censurado"),
    ("Reg Log", "Censurado", "predicciones_logistica_censurado"),
    ("DistilBERT 1 epoch", "Censurado", "predicciones_censurado"),
    ("DistilBERT 2 epoch", "Censurado", "predicciones_censurado_2ep"),
]

tabla = []
for modelo, escenario, variable in modelos_a_evaluar:
    y_pred = globals().get(variable, None)
    clase_f1 = f1_por_clase(y_real, y_pred, etiquetas_clase, modelo, escenario)
    tabla.append({
        "Modelo": modelo,
        "Escenario": escenario,
        "F1 Casa": clase_f1.get("F1 Casa", "N/A"),
        "F1 Departamento": clase_f1.get("F1 Departamento", "N/A"),
        "F1 PH": clase_f1.get("F1 PH", "N/A"),
    })

_df_tabla2 = pd.DataFrame(tabla)[["Modelo", "Escenario", "F1 Casa", "F1 Departamento", "F1 PH"]]
print("Tabla 2: F1 por clase")
display(_df_tabla2)
if y_real is None:
    print("⚠️ df_prueba no está definido en el espacio de ejecución. Ejecuta previamente las celdas de carga y evaluación.")


Tabla 2: F1 por clase


,Modelo,Escenario,F1 Casa,F1 Departamento,F1 PH
0,Bayes,Base,0.8260,0.8667,0.3470
1,SVM,Base,0.9036,0.9499,0.5852
2,Reg Log,Base,0.9080,0.9549,0.5852
3,DistilBERT 1 epoch,Base,0.8739,0.9419,0.6718
4,Bayes,Censurado,0.7993,0.8406,0.3214
5,SVM,Censurado,0.8491,0.9052,0.4244
6,Reg Log,Censurado,0.8682,0.9136,0.4630
7,DistilBERT 1 epoch,Censurado,0.8411,0.8913,0.3791
8,DistilBERT 2 epoch,Censurado,0.8493,0.8918,0.3813


## Explicabilidad global de los modelos

En esta sección mostramos explicaciones globales para los tres modelos clásicos entrenados sobre datos censurados y para DistilBERT con los datos censurados del conjunto de prueba.


In [11]:
import numpy as np
import pandas as pd
from collections import defaultdict

from src.property_text_pipeline import (
    COLUMNA_TEXTO_LIMPIO_CENSURADO,
    entrenar_modelo_base_svm,
    entrenar_modelo_bayes,
    entrenar_modelo_logistica,
)

if "df_entrenamiento" not in globals():
    print("⚠️ No se encuentra df_entrenamiento. Ejecuta primero la celda de preparación de datos.")
else:
    df_train = df_entrenamiento
    y_train = df_train["property_type"]

    def _fit_si_falta(nombre, funcion, columna_texto):
        if globals().get(nombre) is not None:
            return globals()[nombre]
        if columna_texto not in df_train.columns:
            return None
        modelo = funcion(df_train, y_train, columna_texto=columna_texto)
        globals()[nombre] = modelo
        return modelo

    def _mostrar_coeficientes(modelo, nombre):
        if modelo is None:
            print(f"⚠️ No existe el modelo {nombre}.")
            return
        vectorizer = modelo.named_steps["tfidf"]
        clasificador = modelo.named_steps["clasificador"]
        vocab = vectorizer.get_feature_names_out()
        coefs = clasificador.coef_
        clases = clasificador.classes_
        tablas = []
        for idx, clase in enumerate(clases):
            top_idx = np.argsort(coefs[idx])[::-1][:15]
            top_features = vocab[top_idx]
            top_values = coefs[idx][top_idx]
            tablas.append(pd.DataFrame({
                "clase": clase,
                "palabra": top_features,
                "coeficiente": np.round(top_values, 4),
            }))
        resultado = pd.concat(tablas, ignore_index=True)
        display(resultado)

    modelo_reg_log_censurado = _fit_si_falta(
        "modelo_reg_log_censurado",
        entrenar_modelo_logistica,
        COLUMNA_TEXTO_LIMPIO_CENSURADO,
    )

    print("### Regresión Logística censurada: coeficientes por clase")
    _mostrar_coeficientes(modelo_reg_log_censurado, "Regresión Logística")


### Regresión Logística censurada: coeficientes por clase


,clase,palabra,coeficiente
0,Casa,lote,3.8459
1,Casa,chalet,3.4217
2,Casa,terreno,2.8955
3,Casa,garage,2.4381
4,Casa,autos,2.3819
5,Casa,pileta,2.2687
6,Casa,desarrollada,2.2520
7,Casa,galeria,2.1039
8,Casa,fondo,1.8151
9,Casa,hogar,1.7761


In [12]:
if "df_entrenamiento" in globals():
    modelo_svm_censurado = _fit_si_falta(
        "modelo_svm_censurado",
        entrenar_modelo_base_svm,
        COLUMNA_TEXTO_LIMPIO_CENSURADO,
    )
    modelo_bayes_censurado = _fit_si_falta(
        "modelo_bayes_censurado",
        entrenar_modelo_bayes,
        COLUMNA_TEXTO_LIMPIO_CENSURADO,
    )

    print("### SVM censurado: coeficientes por clase")
    _mostrar_coeficientes(modelo_svm_censurado, "SVM")


### SVM censurado: coeficientes por clase


,clase,palabra,coeficiente
0,Casa,chalet,3.1794
1,Casa,lote,2.3985
2,Casa,desarrollada,2.3296
3,Casa,terreno,2.2957
4,Casa,cerrado,1.8834
5,Casa,siguientes,1.8413
6,Casa,electricidad,1.7639
7,Casa,parrillero,1.7636
8,Casa,utilizado,1.7343
9,Casa,pa,1.6280


In [13]:
## EXPLICABILIDAD BAYES
import numpy as np
import pandas as pd

def _mostrar_palabras_discriminantes_bayes(modelo):
    if modelo is None:
        print("⚠️ No existe el modelo Bayes.")
        return

    vectorizer = modelo.named_steps["tfidf"]
    clasificador = modelo.named_steps["clasificador"]
    vocab = vectorizer.get_feature_names_out()
    log_prob = clasificador.feature_log_prob_
    clases = clasificador.classes_

    tablas = []
    for idx, clase in enumerate(clases):
        otros_idx = [i for i in range(len(clases)) if i != idx]
        max_log_prob_otros = np.max(log_prob[otros_idx], axis=0)
        diferencia = log_prob[idx] - max_log_prob_otros
        top_idx = np.argsort(diferencia)[::-1][:15]
        tablas.append(pd.DataFrame({
            "clase": clase,
            "palabra": vocab[top_idx],
            "poder_discriminante": np.round(diferencia[top_idx], 4),
            "log_prob_clase": np.round(log_prob[idx][top_idx], 4),
        }))
    resultado = pd.concat(tablas, ignore_index=True)
    display(resultado)

if 'modelo_bayes_censurado' in globals():
    print("### Naive Bayes Censurado: Palabras con mayor poder discriminante por clase")
    _mostrar_palabras_discriminantes_bayes(modelo_bayes_censurado)
elif 'modelo_bayes' in globals():
    print("### Naive Bayes: Palabras con mayor poder discriminante por clase")
    _mostrar_palabras_discriminantes_bayes(modelo_bayes)


### Naive Bayes Censurado: Palabras con mayor poder discriminante por clase


,clase,palabra,poder_discriminante,log_prob_clase
0,Casa,family,2.1519,-7.4000
1,Casa,lotes,2.0717,-7.2996
2,Casa,laguna,2.0662,-7.0820
3,Casa,huespedes,1.9056,-7.7482
4,Casa,desarrollada,1.8594,-6.3803
5,Casa,muelle,1.8393,-7.9224
6,Casa,comparten,1.7953,-7.2032
7,Casa,venecitas,1.7335,-7.9266
8,Casa,country,1.6587,-7.5018
9,Casa,cpcpi,1.6050,-8.0966


In [19]:
import pandas as pd
from collections import defaultdict

try:
    from transformers_interpret import SequenceClassificationExplainer
except ModuleNotFoundError:
    SequenceClassificationExplainer = None

if SequenceClassificationExplainer is None:
    print("transformers-interpret no esta instalado en este kernel. Instalalo con `uv sync` o `uv pip install transformers-interpret` antes de ejecutar esta celda.")
elif "modelo_censurado" not in globals() or "df_prueba" not in globals() or "tokenizador" not in globals() or "id_a_etiqueta" not in globals():
    print("⚠️ Faltan variables necesarias para la explicabilidad. Asegurate de que modelo_censurado, tokenizador, df_prueba e id_a_etiqueta existan.")
else:
    explainer = SequenceClassificationExplainer(
        modelo_censurado,
        tokenizador,
        custom_labels=id_a_etiqueta
    )

    importancia_por_clase = defaultdict(lambda: defaultdict(float))

    muestra = df_prueba.sample(n=100, random_state=42)

    for texto in muestra['texto_limpio']:
        tokens = tokenizador.encode(str(texto), add_special_tokens=False, truncation=True, max_length=500)
        texto_truncado = tokenizador.decode(tokens)
        atribuciones = explainer(texto_truncado)

        # Obtenemos el índice numérico seguro y lo mapeamos a su nombre real
        clase_predicha_idx = explainer.predicted_class_index
        if hasattr(clase_predicha_idx, "item"):
            clase_predicha_idx = int(clase_predicha_idx.item())
        else:
            clase_predicha_idx = int(clase_predicha_idx)
        clase_predicha = id_a_etiqueta[clase_predicha_idx]

        for palabra, score in atribuciones:
            if palabra not in tokenizador.all_special_tokens and len(palabra) > 2:
                # Solo sumamos los scores positivos (los que empujan hacia esa clase)
                if score > 0:
                    importancia_por_clase[clase_predicha][palabra] += score

    # Iteramos automáticamente sobre todas las clases encontradas
    for clase, importancias in importancia_por_clase.items():
        top_palabras = sorted(importancias.items(), key=lambda x: x[1], reverse=True)[:15]
        df_top = pd.DataFrame(top_palabras, columns=["Palabra", "Score Positivo Acumulado"])
        print(f"### Top 15 palabras representativas - {clase}")
        display(df_top)


### Top 15 palabras representativas - Departamento


,Palabra,Score Positivo Acumulado
0,bal,20.452898
1,edificio,15.696280
2,departamento,5.140006
3,placa,3.876800
4,##con,3.300857
5,##s,3.023956
6,##o,2.844252
7,torre,2.803258
8,ascenso,2.245465
9,##fre,2.004188


### Top 15 palabras representativas - Casa


,Palabra,Score Positivo Acumulado
0,planta,8.365226
1,casa,6.845928
2,lot,5.609805
3,jardin,4.613917
4,##e,3.133840
5,auto,2.673056
6,terreno,2.655631
7,##let,2.426541
8,##s,1.857124
9,##mosa,1.850048


### Top 15 palabras representativas - PH


,Palabra,Score Positivo Acumulado
0,entrada,3.831561
1,##plex,2.633696
2,ref,1.438881
3,independiente,1.230687
4,planta,1.054788
5,casa,0.860448
6,codi,0.717696
7,puede,0.645819
8,fondo,0.529249
9,block,0.417069
